<a href="https://colab.research.google.com/github/mlorenzon95-ops/lepi-bakery-oos-analyzer/blob/main/lepi_out_of_stock_analyser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🛠️ 1. Setup & File Upload (Version 4)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from matplotlib.colors import ListedColormap
from datetime import timedelta
from google.colab import files

# Silence technical warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

print("Step 1: Please select your Excel file (e.g., oos_data_2026.xlsx):")
uploaded = files.upload()

# Capture the uploaded file name
if uploaded:
    FILE_NAME = list(uploaded.keys())[0]
    print(f"✅ File '{FILE_NAME}' uploaded successfully!")
else:
    print("❌ No file selected. Please run this cell again.")

Step 1: Please select your Excel file (e.g., oos_data_2026.xlsx):


In [8]:
# @title ⚙️ 2. Analysis Engine (Version 4)
class SeniorRetailAnalyzer:
    def __init__(self, file_path):
        self.file_path = file_path

    def load_and_sync_data(self, sheet_name):
        """Loads Catalog and Logs, syncing every item in the inventory."""
        try:
            xl = pd.ExcelFile(self.file_path)
            sheet_names = xl.sheet_names
            cat_sheet_name = next((s for s in sheet_names if s.strip().upper() == 'CATALOG'), sheet_names[0])
            catalog_raw = pd.read_excel(self.file_path, sheet_name=cat_sheet_name)

            full_inventory = {}
            for col in catalog_raw.columns:
                cat_name = str(col).strip().title()
                items = catalog_raw[col].dropna().unique()
                full_inventory[cat_name] = [str(i).strip() for i in items]

            target_sheet = next((s for s in sheet_names if s.strip().upper() == sheet_name.upper()), None)
            if not target_sheet:
                print(f"❌ Error: Sheet '{sheet_name}' not found.")
                return None, None

            df_logs = pd.read_excel(self.file_path, sheet_name=target_sheet)
            df_logs.columns = [str(c).strip().title() for c in df_logs.columns]
            existing_cols = [c for c in ['Bread', 'Pastry', 'Sandwiches'] if c in df_logs.columns]

            melted = df_logs.melt(id_vars=['Date'], value_vars=existing_cols, var_name='Category_Log', value_name='Item_Name')
            melted = melted.dropna(subset=['Item_Name'])
            melted = melted.assign(Item_Name=melted['Item_Name'].astype(str).str.split(',')).explode('Item_Name')
            melted['Item_Name'] = melted['Item_Name'].str.strip()
            melted['Date'] = pd.to_datetime(melted['Date'], dayfirst=True, errors='coerce')
            melted = melted.dropna(subset=['Date'])
            melted['Weekday'] = melted['Date'].dt.day_name()
            melted['Week_Number'] = melted['Date'].dt.isocalendar().week
            return melted, full_inventory
        except Exception as e:
            print(f"❌ Error during processing: {e}")
            return None, None

    def plot_and_save_dashboards(self, df_logs, inventory):
        """Generates heatmaps, saves them, and triggers auto-download."""
        from google.colab import files as colab_files
        curr_week = df_logs['Week_Number'].max()
        week_data = df_logs[df_logs['Week_Number'] == curr_week]
        any_date_in_week = week_data['Date'].iloc[0]
        monday_of_week = any_date_in_week - timedelta(days=any_date_in_week.weekday())
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        date_labels = [f"{day}\n{(monday_of_week + timedelta(days=i)).strftime('%d/%m')}" for i, day in enumerate(day_order)]
        traffic_cmap = ListedColormap(['#f8f9f9', '#ff0000'])

        for cat in ['Bread', 'Pastry', 'Sandwiches']:
            all_items_in_cat = inventory.get(cat, [])
            if not all_items_in_cat: continue
            matrix = pd.DataFrame(0, index=all_items_in_cat, columns=day_order)
            cat_logs = week_data[week_data['Item_Name'].isin(all_items_in_cat)]
            if not cat_logs.empty:
                counts = pd.crosstab(cat_logs['Item_Name'], cat_logs['Weekday'])
                matrix.update(counts)
            matrix['Total'] = matrix.sum(axis=1)
            matrix = matrix.sort_values(by='Total', ascending=False).drop(columns='Total')

            plt.figure(figsize=(14, max(4, len(matrix) * 0.6)))
            ax = sns.heatmap(matrix, annot=False, cmap=traffic_cmap, cbar=False, linewidths=1, linecolor='#d1d1d1', vmin=0, vmax=1)
            plt.title(f"RETAIL MONITOR: {cat.upper()} | Week {curr_week}\nRed = Stock-Out, White = Available", fontsize=16, fontweight='bold', loc='left', pad=25)
            ax.set_xticklabels(date_labels, fontsize=11, fontweight='bold')
            plt.yticks(rotation=0)
            plt.tight_layout()

            filename = f"OOS_Report_{cat}_Week_{curr_week}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            try:
                colab_files.download(filename)
                print(f"💾 File downloaded: {filename}")
            except:
                print(f"💾 File saved in sidebar: {filename}")
            plt.show()

In [ ]:
# @title 🚀 3. Interactive Analysis Dashboard (Version 4)
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create an output area for charts
output_area = widgets.Output()

def on_run_clicked(b):
    with output_area:
        clear_output() # Keeps the dashboard clean
        selected_month = month_dropdown.value
        try:
            # Defensive file path check
            actual_path = FILE_NAME[0] if isinstance(FILE_NAME, list) else FILE_NAME
            analyzer = SeniorRetailAnalyzer(actual_path)
            logs, inventory = analyzer.load_and_sync_data(selected_month)

            if logs is not None:
                print(f"📊 Analyzing: {selected_month}...")
                analyzer.plot_and_save_dashboards(logs, inventory)
                print(f"✅ Analysis for {selected_month} complete.")
            else:
                print(f"⚠️ Could not load data for {selected_month}.")
        except NameError:
            print("❌ ERROR: Please run Cell 1 and Cell 2 first!")
        except Exception as e:
            print(f"❌ Error: {e}")

# --- INTERFACE GENERATION ---
try:
    actual_path = FILE_NAME[0] if isinstance(FILE_NAME, list) else FILE_NAME
    xl = pd.ExcelFile(actual_path)
    # Get all sheets EXCEPT Catalog
    month_options = [s for s in xl.sheet_names if s.strip().upper() != 'CATALOG']

    month_dropdown = widgets.Dropdown(
        options=month_options,
        description='Select Month:',
        style={'description_width': 'initial'}
    )

    run_button = widgets.Button(
        description='Run Weekly Analysis',
        button_style='success', # Green color
        icon='check'
    )

    run_button.on_click(on_run_clicked)

    print(f"📂 Connected to: {actual_path}")
    print("Choose the month and click the button. Note: If you upload a NEW file, re-run this cell.")
    display(widgets.VBox([month_dropdown, run_button, output_area]))

except NameError:
    print("❌ ERROR: No file detected. Run Cell 1 and upload your file.")
except Exception as e:
    print(f"❌ Error reading file: {e}")